# **4.9 Environmental and Distribution Shifts**
So far, we’ve explored many practical applications of machine learning, fitting models to various datasets. However, have we ever stopped to consider where the data originally came from? And how do we plan to handle the model’s output in the end? Typically, developers have some data and are eager to build a model without giving much thought to these fundamental questions.<br><br>
Many failed machine learning deployments (i.e., real-world applications) can be traced back to this approach. Sometimes, a model performs exceptionally well when measured by its accuracy on the test set. However, when the data distribution suddenly changes, the model suffers a catastrophic failure in production. More insidiously, sometimes the deployment of the model itself acts as a catalyst that disrupts the data distribution. Let’s consider an example that’s a bit absurd but could very well be real. Suppose we’ve trained a model to predict the default risk of loan applicants—specifically, who will repay their loans and who will default. This model discovers a correlation between an applicant’s footwear and default risk (applicants wearing Oxford shoes will repay, while those wearing sneakers will default). Consequently, the model might be inclined to approve loans for all applicants wearing Oxford shoes and reject all those wearing sneakers.<br><br>
This situation could have disastrous consequences. First, once the model begins making decisions based on footwear, customers will realize this and adjust their behavior. Before long, all applicants will be wearing Oxford shoes, yet their creditworthiness will not improve accordingly. **In short, similar problems exist in many machine learning applications: by introducing model-based decisions into the environment, we may end up undermining the model itself**.

## **4.9.1 Types of Distribution Shifts**
First, we consider the various ways in which data distributions may change, as well as the measures that can be taken to preserve model performance. In a classic scenario, suppose the training data is sampled from a distribution $p_S(x,y)$, but the test data will contain unlabeled samples drawn from a different distribution $p_T(x,y)$. **A sobering reality is that it is impossible to learn a classifier without any assumptions about the relationship between $p_S$ and $p_T$.**<br><br>
Consider a binary classification problem: distinguishing between dogs and cats. If the distribution can be shifted in any way, then our scenario allows for a pathological case in which the input distribution remains unchanged—$p_S(x) = p_T(x)$—but the labels are completely reversed—$p_S(y|x) = 1 − p_T(y|x)$. In other words, all the “cats” from the future are now dogs, and what we previously called “dogs” are now cats. Since the distribution of the input $p(x)$ remains unchanged in this case, we cannot
distinguish this scenario from one in which the distribution has not changed at all.

### **Covariate Bias**
Among the various types of distribution shifts, covariate shift is perhaps the most widely studied. Here, we assume that while the distribution of the input may change over time, the labeling function (i.e., the conditional distribution $P(y|x)$) remains unchanged. Statisticians refer to this as a covariate shift because the problem arises from changes in the distribution of the covariates (features). Although we can sometimes infer distribution shifts without invoking causality, a covariate shift is a natural assumption when we believe that $x$ causes $y$.<br><br>
Consider the problem of distinguishing between cats and dogs: the training data includes the images shown in [Figure 4.9.1](#fig491).
##### **Fig(4.9.1)**
![Training Data for Distinguishing Cats from Dogs](../../images/picture4_9_1.png)<br><br>
During the test, we were asked to classify the images shown in [Figure 4.9.2](#fig492).
##### **Fig(4.9.2)**
![Testing Data for Distinguishing Cats from Dogs](../../images/picture4_9_2.png)<br><br>
The training set consists of real photographs, while the test set contains only cartoon images. Suppose training is performed on a dataset whose features are fundamentally different from those of the test set; if there is no way to adapt to the new domain, problems may arise.

### **Label Offset**
Label shift describes a problem that is the opposite of covariate shift. Here, we assume that the label marginal probability P(y) can change, but the class conditional distribution P(x|y) remains constant across different domains. Label shift is a reasonable assumption when we consider that y causes x. For example, when predicting a patient’s disease, we might make a judgment based on symptoms, even if the relative prevalence of the disease changes over time. Label shift is an appropriate assumption here because the disease causes the symptoms. In other cases, both the label shift and covariate shift assumptions may hold simultaneously. For example, when the label is certain, the covariate shift assumption is satisfied even if y causes x. Interestingly, in these situations, it is generally advantageous to use methods based on the label shift assumption. This is because these methods tend to incorporate features that resemble labels (typically low-dimensional) rather than features that resemble inputs (typically high-dimensional).

### **Concept Shift**
We may also encounter “concept shifts”: this issue arises when the definition of a label changes. It sounds strange—a cat is just a cat, isn’t it? However, other categories do change over time as their usage evolves. Diagnostic criteria for mental illnesses, so-called “trends,” and job titles, among other things, are all everyday examples of concept shifts. As it turns out, if we were to travel across the United States and adjust our data sources based on our geographic location, we would discover a significant conceptual shift in the distribution of names for “soft drinks,” as shown in [Figure 4.9.3](#fig493).
##### **Fig(4.9.3)**
![Conceptual Shifts in the Names of U.S. Soft Drinks](../../images/picture4_9_3.png)<br><br>
If we were to build a machine translation system, the distribution of $P(y | x)$ might yield different translations depending on our location. This issue may be difficult to detect. Therefore, it would be best to make use of knowledge that gradually shifts over time or space.

## **4.9.2 Examples of Distribution Shifts**
Before delving into formal systems and algorithms, we can discuss some specific cases where covariate shifts or conceptual shifts may not be immediately apparent.

### **Medical Diagnosis**
Suppose we want to design an algorithm to detect cancer. We collect data from healthy people and patients, and then train the algorithm. It works very well and achieves high accuracy, so we conclude that it’s ready to succeed in medical diagnosis. But let’s not jump to conclusions just yet.<br><br>
The distribution of training data may differ significantly from the distribution of data encountered in real-world settings. This happened to an unfortunate startup with which some of us authors collaborated a few years ago. They were developing a blood test primarily targeting a disease that affects older men and hoped to use blood samples collected from patients for their research. However, obtaining blood samples from healthy men proved much more difficult than obtaining them from patients already in their system. To compensate, the startup solicited blood donations from students on a university campus to serve as healthy control samples for developing the test. The startup then asked us if we could help them build a classifier to detect the disease.<br><br>
As we explained to them, it is indeed easy to distinguish between healthy and diseased populations with near-perfect accuracy. However, this may be due to differences among the subjects in age, hormone levels, physical activity, diet, alcohol consumption, and many other factors unrelated to disease. This may not apply to classifiers designed to detect disease. These samples may be subject to extreme covariate bias. Furthermore, it is unlikely that this situation can be corrected using conventional methods. In short, they wasted a lot of money.

### **Self-Driving Cars**
For a company looking to use machine learning to develop self-driving cars, a key component is the “curb detector.” Since obtaining real, annotated data is very costly, they came up with a “clever” idea: to use synthetic data from a game rendering engine as supplementary training data. This worked very well for the “test data” extracted from the rendering engine, but when applied to a real car, it was a complete disaster. As it turned out, the curbs were rendered as a very simple texture. More importantly, all curbs were rendered with the exact same texture, and the curb detector quickly learned this “feature.”<br><br>
Something similar happened when the U.S. military first attempted to detect tanks in a forest. They took aerial photographs of the forest without any tanks present, then drove tanks into the forest and took another set of photographs. The classifier trained using these two sets of data seemed to work quite well. Unfortunately, the classifier had merely learned to distinguish between trees in the shade and those out of the shade: the first set of photographs was taken in the early morning, while the second was taken at noon.

### **Non-stationary distribution**
When the distribution changes slowly and the model has not been sufficiently updated, a more nuanced situation arises: a nonstationary distribution. Here are some typical examples:<br>
   * Training a computational advertising model but not updating it frequently (for example, a model trained in 2009 would not know about an unknown new device called the iPad that has just been released);
   * Set up a spam filter that effectively detects all spam. However, spammers have become more sophisticated and are creating new messages that don't look like any spam we've seen before;
   * Build a product recommendation system that works throughout the winter but continues to recommend Santa hats long after Christmas is over.

### **More Anecdotes**
   * Build a face detector that performs well on all benchmark tests but fails on the test data: problematic examples include close-ups where the face fills the entire image (there is no such data in the training set).
   * We have developed a web search engine for the U.S. market and hope to deploy it in the United Kingdom.
   * Train an image classifier on a large dataset in which the number of instances per major category is roughly equal across the dataset—for example, 1,000 categories, each represented by 1,000 images. However, when this system is deployed in the real world, the actual distribution of photo labels is clearly uneven.

## **4.9.3 Distribution Offset Correction**
As we have discussed, in many cases the training and test distributions P(x,y) are different. In some cases, we are fortunate that the model works well regardless of any shifts in the covariates, labels, or concepts. In other cases, we can perform better by employing strategies to address these shifts. The remainder of this section will focus on the technical details of addressing these shifts.

### **Expected Risk and Actual Risk**
First, let’s reflect on what actually happens during model training. The features and corresponding labels of the training data {$(x1, y1), ..., (xn, yn)$} are processed iteratively, and the parameters of the model $f$ are updated after each mini-batch. For simplicity, we will not consider regularization, which significantly reduces the training loss:
##### **(4.9.1)**
$$\underset{f}{minimize} \frac{1}{n} \sum\limits_{i=1}^{n} l(f(x_i), y_i)$$
Here, $l$ is the loss function, which measures how “bad” the prediction $f(x_i)$ is given the label $y_i$. Statisticians refer to this term in [(4.9.1)](#491) as the empirical risk. The empirical risk is intended to approximate the true risk—the average loss over the entire training dataset, that is, the expected value of the total loss for all data drawn from its true distribution $p(x,y)$:
##### **(4.9.2)**
$$E_{p(x,y)}[l(f(x),y)]=\iint l(f(x),y)p(x,y)dxdy$$
However, in practice, we are usually unable to obtain population data. Therefore, empirical risk minimization—that is, minimizing the empirical risk in [(4.9.1)](#491)—is a practical machine learning strategy that aims to approximate the minimization of the true risk.

### **Covariate Offset Correction**
Suppose we want to estimate $P(y|x)$ for labeled data $(xi, yi)$. However, the observations $x_i$ are drawn from a source distribution $q(x)$, rather than from the target distribution $p(x)$. **Fortunately, the dependency assumption implies that the conditional distribution remains unchanged, that is: $p(y|x) = q(y|x)$.** If the source distribution $q(x)$ is “incorrect,” we can correct for this by using the following simple identity in the calculation of the true risk:
##### **(4.9.3)**
$$\iint l(f(x),y)p(y|x)p(x)dxdy = \iint l(f(x),y)q(y|x)q(x)\frac{p(x)}{q(x)}dxdy$$
In other words, we need to **reweight each data sample based on the ratio of the probability that the data comes from the correct distribution to the probability that it comes from the incorrect distribution**:
##### **(4.9.4)**
$$\beta_{i} \overset{\mathrm{def}}{=} \frac{p(x_i)}{q(x_i)}$$
By substituting the weights $\beta_i$ into each data sample $(xi, yi)$, we can use “weighted empirical risk minimization” to train the model:
##### **(4.9.5)**
$$\underset{f}{minimize} \frac{1}{n} \sum\limits_{i=1}^{n} \beta_i l(f(x_i), y_i)$$
Since we do not know this ratio, we need to estimate it. There are many methods available, including some sophisticated operator theory approaches that attempt to recalibrate the expected operator directly using the minimum norm or maximum entropy principles. For any such method, we need to draw samples from two distributions: the “true” distribution $p$, obtained by examining the test data; and the training set q, which is easily obtained through synthetic generation. Note that we only need the features $x \sim p(x)$; we do not need access to the labels $y \sim p(y)$.<br><br>
For simplicity, let’s assume we draw the same number of samples from two distributions, $p(x)$ and $q(x)$. Now, let’s use the label $z$: data drawn from $p$ is assigned a value of 1, and data drawn from $q$ is assigned a value of −1. The probability in the mixed dataset is then given by the following equation:
##### **(4.9.6)**
$$P(z=1|x)=\frac{p(x)}{p(x)+q(x)} \; and \; hence \frac{P(z=1|x)}{P(z=-1|x)} = \frac{p(x)}{q(x)}$$
Therefore, if we use a log-likelihood regression method in which $P(z=1|x) = \frac{1}{1+exp(−h(x))}$ (where $h$ is a parametric function), it naturally follows
that:
$$\beta_{i}=\frac{1/(1+exp(-h(x_i)))}{exp(-h(x_i))/(1+exp(-h(x_i)))}$$
Therefore, we need to address two issues: the first concerns distinguishing between data from the two distributions; the second concerns minimizing the weighted empirical risk in ([4.9.5](#495)). In this problem, we will weight the terms by $\beta_i$.<br><br>
Now, let’s take a look at the complete algorithm for correcting covariate bias. Suppose we have a training set {$(x_1, y_1), ..., (x_n, y_n)$} and an unlabeled test set {$u_1, ..., u_x$}. For covariate bias, we assume that $x_i$ (for 1 ≤ $i$ ≤ n) comes from a source distribution and $u_i$ comes from the target distribution. The following is a typical algorithm for correcting covariate bias:
   1. Generate a binary classification training set: {$(x_1, −1), ..., (x_n, −1), (u_1, 1), ..., (u_x, 1)$}.
   2. Train a binary classifier using log-likelihood regression to obtain the function $h$.
   3. We weight the training data using $\beta_i = exp(h(x_i))$ or, better yet, $\beta_i = min(exp(h(x_i)), c)$ (where $c$ is a constant).
   4. Train {$(x_1, y_1), ..., (x_n, y_n)$} in ([4.9.5](#495)) using the weights βi.<br><br>
**Please note that the above algorithm relies on an important assumption: each data sample in the target distribution (e.g., the test distribution) must have a nonzero probability of occurring during training. If we find a point where $p(x) > 0$ but $q(x) = 0$, the corresponding importance weight will be infinite.**

### **Label Offset Correction**
Suppose we are dealing with a classification task with k classes. Using the same notation as in [Section 4.9.3](./concise_impl_of_mlp.ipynb#43-a-concise-implementation-of-a-multi-layer-perceptron), $q$ and $p$ denote the source distribution (e.g., the distribution during training) and the target distribution (e.g., the distribution during testing), respectively. Suppose the distribution of labels changes over time: $q(y) ≠ p(y)$, but the class-conditional distribution remains constant: $q(x | y) = p(x | y)$. If the source distribution $q(y)$ is “incorrect,” we can correct it based on the identity in the true risk defined in ([4.9.2](#492)):
##### **(4.9.8)**
$$\iint l(f(x),y)p(x|y)p(y)dxdy = \iint l(f(x),y)q(x|y)q(y)\frac{p(y)}{q(y)}dxdy$$
Here, the importance weights will correspond to the label likelihood ratios
##### **(4.9.9)**
$$\beta_i \overset{\mathrm{def}}{=} \frac{p(y_i)}{q(y_i)}$$
**One advantage of label shifting is that if we have a reasonably good model for the source distribution, we can obtain consistent estimates of these weights without having to deal with the other dimensions in the surrounding space.** In deep learning, inputs are often high-dimensional objects (such as images), while labels are typically low-dimensional (such as categories).<br><br>
To estimate the target label distribution, we first use a ready-made classifier with reasonably good performance (typically trained on the training data) and compute its confusion matrix using the validation set (which also comes from the training distribution). The confusion matrix $\mathbb{C}$ is a $k×k$ matrix, where each column corresponds to a label category and each row corresponds to a category predicted by the model. **The value of each cell, $c_{ij}$, represents the proportion of samples in the validation set where the true label is $j$ but the model predicts $i$.**<br><br>
At this point, we cannot directly compute the confusion matrix on the target data because we cannot see the labels of the samples in the real-world setting, unless we set up a complex real-time annotation process. However, what we can do is take the average of all models’ predictions during testing to obtain the average model output $µ(\hat{y})$ \in $\mathbb{R}_k$, where the i-th element $\mu (\hat{y})$ is the total predicted score for sample $i$ in the test set.<br><br>
The results show that if our classifier is reasonably accurate from the outset, the target data contains only categories we have seen before, and the label shift assumption holds (the strongest assumption here), we can estimate the label distribution of the test set by solving a simple linear system
##### **(4.9.10)**
$$Cp(y)= \mu (\hat{y})$$
As an estimate, $\sum_{j=1}^{k}c_{ij}p(y_j)= \mu (\hat{y}_{i})$ holds for all $1 \le i \le k$, where $p(y_j)$ is the j^{th} element of the $k$-dimensional label distribution vector $p(y)$. If our classifier is sufficiently accurate from the outset, then the confusion matrix C will be invertible, and we can thus obtain a solution $p(y) = C^{−1}\mu (\hat{y})$.<br><br>
Since we observe the labels on the source data, it is easy to estimate the distribution $q(y)$. Then, for any training sample $i$ with label $y_i$, we can use our estimate of the ratio $p(y_i)/q(y_i)$ to compute the weight $\beta_i$ and substitute it into the weighted empirical risk minimization in ([4.9.5](#495)).

### **Conceptual Shift Correction**
**Concept drift is difficult to address in a principled manner.** For example, if a problem suddenly shifts from “distinguishing cats from dogs” to “distinguishing white animals from black animals,” **there is no solution other than to start from scratch by collecting new labels and retraining the model**. Fortunately, such extreme shifts are rare in practice. **On the contrary, changes in concepts are typically gradual**. Here are a few examples:
   * In programmatic advertising, once a new product is launched, the old product becomes less popular. This means that ad distribution and popularity change gradually, and any click-through rate predictor needs to adapt accordingly;
   * Due to environmental wear and tear, the lenses of traffic cameras gradually deteriorate, affecting the image quality of the cameras;
   * News content is constantly changing (i.e., new stories are emerging).<br>

In this situation, we can use the same method as when training the network to adapt it to changes in the data. In other words, we update the existing network weights using the new data, rather than training the network from scratch.

## **4.9.4 Classification of Learning Problems**
Armed with our understanding of how to handle changes in distributions, we can now consider other aspects of formalizing machine learning problems.

### **Batch Learning**
In batch learning, we have access to a set of training features and labels {$(x_1, y_1), ..., (x_n, y_n)$}, which we use to train $f(x)$. We then deploy this model to score new data (x, y) drawn from the same distribution. For example, we can train a cat detector using a large number of images of cats and dogs. Once trained, we deploy it as part of a smart cat-door computer vision system designed to allow only cats to enter. This system is then installed in customers’ homes and, for the most part, will never need to be updated.

### **Online Learning**
In addition to learning “in bulk,” we can also learn data points (xi, yi) individually “online.” More specifically, we first observe xi, then derive an estimate f(xi), and only after doing so do we observe yi. Depending on our decision, we then receive a reward or incur a loss. Many real-world problems fall into this category. For example, we need to predict tomorrow’s stock prices so that we can trade based on that prediction. At the end of the day, we evaluate whether our prediction was profitable. In other words, in online learning, we follow the cycle described below. In this cycle, given new observations, we continuously improve our model.
##### **(4.9.11)**
$$model\,f_t \rightarrow\, estimate\,f_t(x_t) \rightarrow\, observation\, y_t \rightarrow\, loss\, l(y_t, f_t(x_t)) \rightarrow\, model\, f_{t+1}$$

### **Slot Machines**
Slot machines (bandits) are a special case of the problem described above. While in most learning problems we have a continuously parameterized function f (e.g., a deep neural network), in a slot machine problem we have only a finite number of levers to pull. **In other words, the actions we can take are finite. It is not surprising that stronger optimality guarantees can be obtained for this simpler problem.** We include it here primarily because this problem is often viewed as a specific scenario of a general learning problem.

### **Control**
In many cases, the environment remembers what we do. Not necessarily in an adversarial way, but it does remember, and its response will depend on what happened previously. For example, a coffee boiler controller will observe different temperatures depending on whether the boiler was heated previously. In this case, the PID (Proportional–Integral–Derivative) controller algorithm is a popular choice. Similarly, a user’s behavior on a news website will depend on the content previously shown to her (for example, she typically reads most news articles only once). Many such algorithms form an environmental model in which their behavior makes their decision-making appear less random. In recent years, control theory (such as variants of PID) has also been used to automatically tune hyperparameters to achieve better deconstruction and reconstruction quality, thereby improving the diversity of generated text and the reconstruction quality of generated images (Shao et al., 2020).

### **Reinforcement Learning**
Reinforcement learning emphasizes how to act based on the environment in order to maximize expected utility. Chess, Go, backgammon, and StarCraft are all examples of reinforcement learning applications. Other examples include designing a controller for an autonomous vehicle or otherwise responding to the driving behavior of an autonomous vehicle (for example, attempting to avoid an object, attempting to cause an accident, or attempting to cooperate with it).

### **Taking the environment into account**
**A key distinction between the scenarios described above is that a strategy that may be consistently effective in a static environment may not always be effective when the environment is subject to change.** For example, an arbitrage opportunity identified by a trader is likely to disappear by the time the trader begins to exploit it. The speed and nature of environmental changes largely determine the types of algorithms we can employ. For instance, if we know that conditions will change only slowly, we can ensure that any estimates also change only slowly. If we know that the environment may change instantaneously, but such changes are very rare, we can take this into account when using algorithms. This type of knowledge is crucial when a data scientist is trying to solve a problem that changes over time.

## **4.9.5 Fairness, Accountability, and Transparency in Machine Learning**
Finally, it is important to recognize that when we deploy machine learning systems, we are not merely optimizing a predictive model; rather, we are typically providing a tool that will be used to make automated decisions—either partially or entirely. These technological systems have the potential to impact everyone’s lives through the decisions they make.<br><br>
The leap from consideration of predictions to decision-making raises not only new technical challenges but also a series of ethical issues that must be carefully considered. If we are deploying a medical diagnostic system, we need to know for which populations it may be effective and for which it may be ineffective. Ignoring foreseeable risks to the well-being of a subgroup could lead us to provide substandard care. Furthermore, once we design an entire decision-making system, we must take a step back and reconsider how we evaluate our technology. Among the consequences of this shift in perspective, we find that accuracy rarely serves as an appropriate metric. For example, when we translate predictions into action, we typically take into account the potential cost sensitivity of making errors in various ways. To illustrate: if misclassifying an image into one category might be viewed as racial discrimination, while misclassifying it into another category is harmless, we may need to adjust our thresholds accordingly and take these social values into account when designing decision-making processes. We also need to pay attention to how predictive systems create feedback loops. For example, consider a predictive policing system that assigns patrol officers to areas with predicted high crime rates. It is easy to see how a troubling
pattern might emerge:
   1. Neighborhoods with high crime rates will receive more patrols;
   2. As a result, more criminal activity is observed in these communities, providing training data that can be used in future iterations;
   3. Given the additional positive factors, the model predicts that crime in these communities will increase further;
   4. In the next iteration, the updated model will be more focused on a specific region, which will lead to more crimes being detected, and so on.<br><br>

**Typically, during the model refinement process, the various mechanisms through which a model’s predictions become coupled with the training data remain unexplained—a phenomenon researchers refer to as “runaway feedback loops.”** Furthermore, we must first ask ourselves whether we are solving the right problem. For example, prediction algorithms now play a massive role in mediating the spread of information—should the news individuals see be determined by the Facebook pages they like? These are just a small fraction of the “daunting” ethical dilemmas one might encounter in a career in machine learning.

## **Summary**
  * In many cases, the training set and the test set do not come from the same distribution. This is known as distribution shift.
  * True risk is the expected total loss of all data drawn from the true distribution. However, this data set is usually unavailable. Empirical risk is the average loss over the training data, which is used to approximate the true risk. In practice, we perform empirical risk minimization.
  * Under appropriate assumptions, covariate bias and label bias can be detected and corrected during testing. Failing to account for such bias during testing may pose a problem.
  * In some cases, the environment may “remember” automated actions and respond in surprising ways. When building models, we must take this possibility into account, continue to monitor the real-time system, and remain open to the possibility that our models and the environment may become intertwined in unexpected ways.